# Superfermion Docs — Live Snippet Run

Every Python snippet from `website/content/docs/*.mdx`, as **individual cells in page order**.
Run the prelude cell first, then each page's cells **top-to-bottom** (blocks on a page
build on earlier blocks of the same page).

- Python snippets that need cloud credentials (IBM / IonQ / AWS) are shown commented out.
- Shell (`bash`) and diagram (`text`) blocks are not code — omitted; noted in page headers.


In [3]:
# ═══════════════════════════════════════════════════════════════════
# PRELUDE — same imports the verification harness injects.
# Run this cell FIRST, then run the page cells top-to-bottom.
# ═══════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings('ignore')
import os
os.environ['MPLBACKEND'] = 'Agg'
import numpy as np
from math import pi
import superfermion as sf
import sys

# Jupyter's stdout stream has no .reconfigure() — make it a no-op so doc
# snippets that reconfigure stdout (for Windows consoles) run unchanged.
if not hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure = lambda *a, **k: sys.stdout
print("superfermion", sf.__version__, "| prelude ready — now run pages top-to-bottom")

superfermion 0.1.5 | prelude ready — now run pages top-to-bottom


## Page: `getting-started.mdx`

7 Python block(s) — run top-to-bottom.

In [4]:
# ── getting-started.mdx #2 ──
import superfermion as sf
print(sf.__version__)  # e.g. '0.1.5'

0.1.5


In [5]:
# ── getting-started.mdx #3 ──
import superfermion as sf
import sys

# Unicode diagrams on Windows consoles: reconfigure stdout first
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# Bell state: entangle two qubits
qc = sf.Circuit(2).h(0).cx(0, 1)
print(qc.draw())
# q0: ── H ── ● ──
# q1: ────── X ──

q0: ─  [H]  ─   ●   ─
q1: ─  ───  ─   ⊕   ─


In [6]:
# ── getting-started.mdx #4 ──
result = sf.run(qc, shots=1024)
print(result.counts)  # {'00': ~512, '11': ~512}

{'11': 516, '00': 508}


In [7]:
# ── getting-started.mdx #5 ──
state = sf.simulate(qc)       # returns sf.State (Rust-native)
print(state.numpy())           # [0.707+0j, 0, 0, 0.707+0j]
print(state.entropy())         # 0.0
print(state.purity())          # 1.0

[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
0.0
1.0


In [8]:
# ── getting-started.mdx #6 ──
from superfermion import Hamiltonian, PauliString

H = Hamiltonian([PauliString("ZZ", coeff=1.0)])
energy = H.expectation(state.numpy())
print(energy)  # 1.0

# Or pass the Rust observable format directly to State:
# state.expectation([([3, 3], 1.0, 0.0)])   # ZZ: 0=I 1=X 2=Y 3=Z (LSB-first)

0.9999999999999998


In [9]:
# ── getting-started.mdx #7 ──
theta = sf.param("theta")
qc = sf.Circuit(1).ry(theta, 0)

state = sf.simulate(qc, params={"theta": 0.5})
obs = [([3], 1.0, 0.0)]      # Pauli Z
dag = qc.bind({"theta": 0.5}).to_ir()
grads = state.grad(obs, dag, {"theta": 0.5})

print(grads)  # {"theta": -0.479...}

{}


In [10]:
# ── getting-started.mdx #8 ──
# MPS — for weakly entangled circuits, scales to 200+ qubits
qc_mps = sf.Circuit(8).h(0)
for i in range(7):
    qc_mps = qc_mps.cnot(i, i + 1)
result = sf.run(qc_mps, method="mps", bond_dim=64, shots=1024)
print(result.counts)  # {'00000000': ~512, '11111111': ~512}

# Stabilizer — for Clifford-only circuits, scales to 1000+ qubits
qc_stab = sf.Circuit(100).h(0)
for i in range(99):
    qc_stab = qc_stab.cnot(i, i + 1)
result = sf.run(qc_stab, method="stabilizer", shots=1024)
print(list(result.counts.items())[:2])

# Density matrix — for noisy simulation
qc_noisy = sf.Circuit(1).x(0)
noise = sf.NoiseModel().add_depolarizing(0.01)
result = sf.run(qc_noisy, method="density_matrix", noise_model=noise, shots=0)
print(f"purity = {result.metadata['purity']:.4f}")

{'11111111': 518, '00000000': 506}
[('1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 509), ('0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 515)]
purity = 0.9868


## Page: `guides/circuits.mdx`

14 Python block(s) — run top-to-bottom.

In [11]:
# ── guides/circuits.mdx #0 ──
import superfermion as sf

qc = sf.Circuit(3).h(0).cnot(0, 1).cnot(1, 2).measure_all()

In [12]:
# ── guides/circuits.mdx #1 ──
qc = sf.Circuit(2)
qc = qc.h(0)       # Hadamard
qc = qc.x(1)       # Pauli-X (NOT)
qc = qc.y(0)       # Pauli-Y
qc = qc.z(1)       # Pauli-Z
qc = qc.s(0)       # Phase (√Z)
qc = qc.sdg(1)     # S dagger
qc = qc.t(0)       # T (⁴√Z)
qc = qc.tdg(1)     # T dagger
qc = qc.sx(0)      # √X
qc = qc.id(1)      # Identity (no-op, useful as placeholder)

In [13]:
# ── guides/circuits.mdx #2 ──
from math import pi

qc = sf.Circuit(1)
qc = qc.rx(0.5, 0)                       # RX rotation
qc = qc.ry(pi / 2, 0)                    # RY rotation
qc = qc.rz(0.3, 0)                       # RZ rotation
qc = qc.p(pi / 4, 0)                     # Phase gate
qc = qc.u(0.5, 0.3, 0.2, 0)              # Arbitrary U(2) — u(theta, phi, lam, q)

In [14]:
# ── guides/circuits.mdx #3 ──
qc = sf.Circuit(3)
qc = qc.cnot(0, 1)        # CNOT (control=0, target=1)
qc = qc.cx(0, 2)          # CX (alias for CNOT)
qc = qc.cz(1, 2)          # Controlled-Z
qc = qc.cy(0, 1)          # Controlled-Y
qc = qc.swap(0, 2)        # SWAP
qc = qc.iswap(1, 2)       # iSWAP
qc = qc.ecr(0, 1)         # Echoed Cross-Resonance

In [15]:
# ── guides/circuits.mdx #4 ──
qc = sf.Circuit(2)
qc = qc.cp(0.5, 0, 1)             # Controlled phase — cp(phi, control, target)
qc = qc.cu(0.5, 0.3, 0.2, 0, 1)  # Controlled U3 — cu(theta, phi, lam, c, t)
qc = qc.rzz(0.5, 0, 1)           # ZZ rotation
qc = qc.rxx(0.3, 0, 1)           # XX rotation
qc = qc.ryy(0.4, 0, 1)           # YY rotation

In [16]:
# ── guides/circuits.mdx #5 ──
qc = sf.Circuit(3)
qc = qc.ccx(0, 1, 2)      # Toffoli (CCNOT)
qc = qc.toffoli(0, 1, 2)  # Alias for CCX
qc = qc.cswap(0, 1, 2)    # Fredkin (CSWAP)
qc = qc.fredkin(0, 1, 2)  # Alias for CSWAP

In [17]:
# ── guides/circuits.mdx #6 ──
import numpy as np

# Custom 1Q unitary
U1 = np.array([[1, 0], [0, 1j]])  # S gate matrix
qc = sf.Circuit(1).unitary(U1, [0])

# Custom 2Q unitary (4x4 matrix)
U2 = np.eye(4)
qc = sf.Circuit(2).unitary(U2, [0, 1])

In [18]:
# ── guides/circuits.mdx #7 ──
qc = sf.Circuit(3)
qc = qc.measure(0)         # Measure single qubit
qc = qc.measure_all()      # Measure all qubits
qc = qc.barrier(0, 1, 2)   # Compilation barrier (no optimization across)
qc = qc.reset(0)            # Reset qubit to |0⟩

In [19]:
# ── guides/circuits.mdx #8 ──
theta = sf.param("theta")
phi = sf.param("phi")

ansatz = (sf.Circuit(2)
    .ry(theta, 0)
    .ry(phi, 1)
    .cnot(0, 1)
    .rz(theta, 1))

print(ansatz.n_parameters)   # 2
print(ansatz.parameters)     # ['theta', 'phi']

# Bind parameter values
bound = ansatz.bind({"theta": 0.5, "phi": 1.2})
print(bound.n_parameters)    # 0 — all bound

2
['theta', 'phi']
0


In [20]:
# ── guides/circuits.mdx #9 ──
qc = sf.Circuit(3).h(0).cnot(0, 1).cnot(1, 2).measure_all()

print(qc.n_qubits)       # 3
print(qc.gate_count)     # 4
print(qc.depth)          # 4
print(qc.n_cbits)        # 3 (from measure_all)
print(qc.n_parameters)   # 0

# Gate inventory
print(qc.count_ops())    # {'h': 1, 'cnot': 2, 'measure': 3}

3
6
4
3
0
{'H': 1, 'CNOT': 2, 'MEASURE': 3}


In [21]:
# ── guides/circuits.mdx #10 ──
import sys

# Unicode diagrams on Windows consoles: reconfigure stdout first
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

print(qc.draw())
# q0: ─── H ─── ● ─────── M ───
# q1: ───────── X ─── ● ─ M ───
# q2: ─────────────── X ─ M ───

q0: ─  [H]  ─   ●   ─  ───  ─[MEASURE]─  ───  ─  ───  ─
q1: ─  ───  ─   ⊕   ─   ●   ─  ───  ─[MEASURE]─  ───  ─
q2: ─  ───  ─  ───  ─   ⊕   ─  ───  ─  ───  ─[MEASURE]─


In [22]:
# ── guides/circuits.mdx #11 ──
# OpenQASM 3.0
print(qc.to_qasm3())

# Gate list (for serialization) — names are uppercase, no params key when empty
gates = qc.to_gate_list()
# [{"name": "H", "qubits": [0]}, ...]

# JSON round-trip
json_str = qc.to_json()
restored = sf.Circuit.from_json(json_str)

# Unitary matrix (small circuits only — exponential cost)
U = qc.to_unitary()  # 2^n × 2^n complex matrix

# Rust IR (for gradient computation)
dag = qc.to_ir()

OPENQASM 3.0;
qubit[3] q;
bit[3] c;

h q[0];
cx q[0], q[1];
cx q[1], q[2];
c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];



In [23]:
# ── guides/circuits.mdx #12 ──
# From a list of gate specifications
gates = [
    {"name": "h", "qubits": [0], "params": []},
    {"name": "cnot", "qubits": [0, 1], "params": []},
    {"name": "ry", "qubits": [0], "params": [0.5]},
]
qc = sf.Circuit(2)
for g in gates:
    qc = getattr(qc, g["name"])(*g["params"], *g["qubits"])

In [24]:
# ── guides/circuits.mdx #13 ──
# Apply H to all qubits
qc = sf.Circuit(4)
for q in range(4):
    qc = qc.h(q)

# Entanglement chain
qc = sf.Circuit(5).h(0)
for q in range(4):
    qc = qc.cnot(q, q + 1)

print(qc.depth)  # 5

5


## Page: `guides/execution.mdx`

11 Python block(s) — run top-to-bottom.

In [25]:
# ── guides/execution.mdx #0 ──
import superfermion as sf

qc = sf.Circuit(2).h(0).cnot(0, 1)

# Run: counts + state + metadata
result = sf.run(qc, device="cpu", shots=1024)
print(result.counts)   # {'00': 512, '11': 512}
print(result.state)    # sf.State

# Simulate: state only (shots=0 implied)
state = sf.simulate(qc, device="cpu")
print(state.numpy())   # [0.707+0j, 0, 0, 0.707+0j]

{'00': 508, '11': 516}
State(n_qubits=2, method='statevector', device='cpu')
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


In [26]:
# ── guides/execution.mdx #1 ──
# Local simulation
sf.run(qc, device="cpu")    # Rust CPU — Rayon multithreading
# sf.run(qc, device="gpu")  # Rust GPU — CUDA (requires sf-gpu); skipped: no GPU here

# Cloud QPUs — skipped in this notebook (need real credentials):
# from superfermion.devices.ibm import IBMDevice
# from superfermion.devices.ionq import IonQDevice
# ibm = IBMDevice(token="...")
# sf.run(qc, device=ibm("ibm_brisbane"), shots=8192)
# ionq = IonQDevice(api_key="...")
# sf.run(qc, device=ionq("aria-1"), shots=1000)
print(list(sf.run(qc, device="cpu", shots=128).counts.items()))

[('11', 62), ('00', 66)]


In [27]:
# ── guides/execution.mdx #2 ──
result = sf.run(qc, method="statevector", shots=1024)
state = sf.simulate(qc, method="statevector")

In [28]:
# ── guides/execution.mdx #3 ──
result = sf.run(qc, method="mps", bond_dim=64, shots=1024)

In [29]:
# ── guides/execution.mdx #4 ──
clifford = sf.Circuit(100).h(0)
for i in range(99):
    clifford = clifford.cnot(i, i + 1)

result = sf.run(clifford, method="stabilizer", shots=1024)

In [30]:
# ── guides/execution.mdx #5 ──
noise = sf.NoiseModel().add_depolarizing(0.001).add_amplitude_damping(0.002)
result = sf.run(qc, method="density_matrix", noise_model=noise, shots=0)

In [31]:
# ── guides/execution.mdx #6 ──
state = sf.simulate(qc)

# Basic properties
print(state.n_qubits)       # 2
print(state.shape)          # (4,)
print(state.method)         # "statevector"
print(state.device)         # "cpu"

# Extract data
sv = state.numpy()          # complex ndarray
samples = state.sample(1000)  # list[int] — measurement outcomes

# Derived quantities
print(state.entropy())          # Von Neumann entropy (0 = pure)
print(state.purity())           # 1.0 = pure, <1 = mixed
print(state.probabilities())    # Outcome probability distribution

# Expectation values
obs = [([3, 3], 1.0, 0.0)]    # ZZ observable
energy = state.expectation(obs)
print(energy)                  # 1.0

# Gradients — on a parameterized ansatz
ansatz = sf.Circuit(2).ry(sf.param("t"), 0).cnot(0, 1)
state_p = sf.simulate(ansatz, params={"t": 0.5})
dag = ansatz.bind({"t": 0.5}).to_ir()
params = {"t": 0.5}
grads = state_p.grad(obs, dag, params)
print(grads)                   # {'t': ...}

# Operations on states
print(state.fidelity(state))         # 1.0 (self-fidelity)
rdm = state.partial_trace([0])       # Reduced density matrix
qfim = state_p.qfim(dag, params)     # Quantum Fisher Information Matrix
print(qfim)

# Create from numpy
import numpy as np
vec = np.array([1, 0, 0, 0], dtype=complex)
state2 = sf.State.from_numpy(vec, n_qubits=2)
print(state2.numpy())

2
[4]
statevector
cpu
0.0
1.0
[0.5 0.  0.  0.5]
0.9999999999999998
{}
0.9999999999999996
[]
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]


In [32]:
# ── guides/execution.mdx #7 ──
obs = [([3, 3], 1.0, 0.0)]    # ZZ observable
result = sf.run(qc, device="cpu", shots=8192)

# Primary outputs
print(result.counts)           # {'00': 4096, '11': 4096}
print(result.state)            # sf.State
print(result.shots)            # 8192

# Derived
print(result.probabilities)    # {'00': 0.5, '11': 0.5}
print(result.statevector)      # ndarray
print(result.circuit)          # original circuit
print(result.metadata)         # execution metadata

# Methods
energy = result.expectation(obs)       # expectation from state
print(energy)
result.plot()                          # histogram plot (saves to file)

{'11': 4087, '00': 4105}
State(n_qubits=2, method='statevector', device='cpu')
8192
{}
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Circuit(n_qubits=2, depth=2, gates=2)
{'backend': 'rust-cpu', 'n_qubits': 2, 'method': 'statevector'}
0.9999999999999998
2026-08-15 04:29:23,460 | WARNING  | superfermion.results | Matplotlib not installed. Summary:
2026-08-15 04:29:23,462 | INFO     | superfermion.results | {'11': 4087, '00': 4105}


In [33]:
# ── guides/execution.mdx #8 ──
noise = (sf.NoiseModel()
    .add_depolarizing(0.001)           # symmetric depolarizing
    .add_amplitude_damping(0.002)      # T1 relaxation
    .add_phase_damping(0.001)          # pure dephasing
    .add_bit_flip(0.0005)              # bit flip
    .add_two_qubit_depolarizing(0.001) # correlated two-qubit noise
)

qc = sf.Circuit(1).x(0)
result = sf.run(qc, method="density_matrix", noise_model=noise, shots=0)
print(f"purity = {result.metadata['purity']:.4f}")

purity = 15.9747


In [34]:
# ── guides/execution.mdx #9 ──
ansatz = sf.Circuit(2).ry(sf.param("t"), 0).cnot(0, 1)

# Both forms are equivalent:
state = sf.simulate(ansatz, params={"t": 0.5})
state = sf.simulate(ansatz.bind({"t": 0.5}))

In [35]:
# ── guides/execution.mdx #10 ──
from superfermion import MethodError

try:
    # T gate on a stabilizer circuit
    sf.run(sf.Circuit(1).t(0), method="stabilizer")
except RuntimeError as e:
    # Non-Clifford gates cannot be simulated with the stabilizer method.
    # (Other backends may raise superfermion.MethodError instead.)
    print(f"Method not applicable: {e}")

Method not applicable: Circuit contains non-Clifford gates — cannot use method='stabilizer'.
  Use method='statevector' or method='mps' instead.


## Page: `guides/compiler.mdx`

11 Python block(s) — run top-to-bottom.

In [36]:
# ── guides/compiler.mdx #0 ──
import superfermion as sf
from superfermion.compiler.specs import HardwareSpec

qc = sf.Circuit(5).h(0).cnot(0, 3).cnot(1, 4).swap(2, 3)

# Optimization only
optimized = sf.compile(qc, level=1)
print(f"Before: {qc.gate_count} gates, After: {optimized.gate_count} gates")

# Compile for a specific device
ibm_target = HardwareSpec(
    name="ibm_device",
    n_qubits=5,
    native_gates=["rz", "sx", "x", "cx"],
    coupling_map=[
        (0, 1), (1, 0), (1, 2), (2, 1), (2, 3),
        (3, 2), (3, 4), (4, 3),
    ],
)
compiled = sf.compile(qc, level=2, target=ibm_target)

Before: 4 gates, After: 6 gates


In [37]:
# ── guides/compiler.mdx #1 ──
import superfermion as sf

# Gate cancellation is part of level-1 optimization
qc = sf.Circuit(2).h(0).h(0).cnot(0, 1).cnot(0, 1)
print(f"Before: {qc.gate_count} gates")

result = sf.compile(qc, level=1)
print(f"After cancellation: {result.gate_count} gates")  # 0

Before: 4 gates
After cancellation: 0 gates


In [38]:
# ── guides/compiler.mdx #2 ──
from superfermion.compiler import apply_noise_suppression

qc = sf.Circuit(1).rz(0.3, 0).rz(0.7, 0).rz(-0.2, 0)
result = sf.compile(qc, level=1)
print(f"After merging: {result.gate_count} gates")  # ~1 gate

After merging: 1 gates


In [39]:
# ── guides/compiler.mdx #3 ──
from superfermion.compiler import BasisTranslationPass

# Decompose H into RZ+RX basis
qc = sf.Circuit(1).h(0)
result = BasisTranslationPass(["RZ", "RX"]).run(qc)
print(f"H decomposed to: {result.gate_count} native gates")  # 3 gates

H decomposed to: 3 native gates


In [40]:
# ── guides/compiler.mdx #4 ──
qc = sf.Circuit(2).swap(0, 1)
result = sf.compile(qc, level=1)
# SWAP → 3 CNOTs + single-qubit gates

In [41]:
# ── guides/compiler.mdx #5 ──
from superfermion.compiler import PauliTwirlingPass

qc = sf.Circuit(2).cnot(0, 1)
twirled = PauliTwirlingPass().run(qc)

In [42]:
# ── guides/compiler.mdx #6 ──
from superfermion.compiler import DynamicalDecouplingPass

dd_pass = DynamicalDecouplingPass(sequence="xy4")
result = dd_pass.run(qc)

In [43]:
# ── guides/compiler.mdx #7 ──
from superfermion.compiler.specs import HardwareSpec

# Linear topology
linear = HardwareSpec(
    name="linear_5",
    n_qubits=5,
    native_gates=["rz", "sx", "x", "cx"],
    coupling_map=[(i, i+1) for i in range(4)] + [(i+1, i) for i in range(4)],
)

# Route onto the linear topology
qc = sf.Circuit(5).h(0).cnot(0, 4).cnot(1, 3).swap(2, 3)
compiled = sf.compile(qc, level=2, target=linear)
print(f"SWAPs inserted for routing: {compiled.gate_count - qc.gate_count}")

SWAPs inserted for routing: 10


In [44]:
# ── guides/compiler.mdx #8 ──
from superfermion.compiler.specs import get_spec, HardwareSpec

# IBM Eagle (127 qubits, heavy-hex)
ibm = get_spec("ibm_eagle")
print(f"ibm_eagle: {ibm.n_qubits} qubits, {len(ibm.coupling_map)} couplings")

# Rigetti Ankaa (84 qubits, grid)
rigetti = get_spec("rigetti_ankaa")
print(f"rigetti_ankaa: {rigetti.n_qubits} qubits, {len(rigetti.coupling_map)} couplings")

# Custom topology
custom = HardwareSpec(
    name="my_chip",
    n_qubits=4,
    native_gates=["rz", "sx", "x", "cz", "iswap"],
    coupling_map=[(0, 1), (1, 2), (2, 3)],
)
print(f"custom: {custom.n_qubits} qubits, {len(custom.coupling_map)} couplings")

ibm_eagle: 127 qubits, 174 couplings
rigetti_ankaa: 84 qubits, 149 couplings
custom: 4 qubits, 3 couplings


In [45]:
# ── guides/compiler.mdx #9 ──
import superfermion as sf

qc = sf.Circuit(1).x(0)

noise = sf.NoiseModel() \
    .add_depolarizing(0.001) \
    .add_amplitude_damping(0.002) \
    .add_phase_damping(0.001)

result = sf.run(qc, method="density_matrix", noise_model=noise, shots=0)
print(f"purity = {result.metadata['purity']:.4f}")

purity = 8.9840


In [46]:
# ── guides/compiler.mdx #10 ──
qc = sf.Circuit(5).h(0).cnot(0, 1)

# Let sf.run() handle compilation automatically
result = sf.run(qc, target="linear_5", shots=4096)
print(result.counts)  # {'00000': ~2048, '00011': ~2048}

{'00000': 2005, '00011': 2091}


## Page: `guides/gradients.mdx`

11 Python block(s) — run top-to-bottom.

In [47]:
# ── guides/gradients.mdx #0 ──
import superfermion as sf

ansatz = sf.Circuit(2).ry(sf.param("t0"), 0).ry(sf.param("t1"), 1).cnot(0, 1)
obs = [([3, 3], 1.0, 0.0)]          # ZZ observable
params = {"t0": 0.5, "t1": 1.2}

state = sf.simulate(ansatz, params=params)
dag = ansatz.bind(params).to_ir()
grads = state.grad(obs, dag, params)
print(grads)

{}


In [48]:
# ── guides/gradients.mdx #1 ──
from superfermion.qml.gradient.adjoint import adjoint_grad_vector
import numpy as np

# Observable: str / dict / SparsePauliOp / Hamiltonian all work
H = sf.Hamiltonian([sf.PauliString("ZZ", coeff=1.0)])
grads = adjoint_grad_vector(ansatz, H, ["t0", "t1"], np.array([0.5, 1.2]))
print(grads)

[ 0.         -0.93203909]


In [49]:
# ── guides/gradients.mdx #2 ──
from superfermion.qml.gradient.parameter_shift import parameter_shift_grad_vector

grads = parameter_shift_grad_vector(ansatz, H, ["t0", "t1"], np.array([0.5, 1.2]))
print(grads)

[ 0.         -0.93203909]


In [50]:
# ── guides/gradients.mdx #3 ──
from superfermion.qml.gradient.spsa import spsa_grad
import numpy as np

def energy_fn(p):
    state = sf.simulate(ansatz, params=dict(zip(["t0", "t1"], p)))
    return state.expectation(obs)

grads = spsa_grad(energy_fn, np.array([0.5, 1.2]), seed=42, delta=0.01)
print(grads)

[ 0.93202355 -0.93202355]


In [51]:
# ── guides/gradients.mdx #4 ──
from superfermion.qml.gradient.qng import qng_step

state = sf.simulate(ansatz, params=params)
dag = ansatz.to_ir()   # unbound DAG for the QFIM metric
new_params = qng_step(state, dag, obs, ["t0", "t1"], params, learning_rate=0.1)
print(new_params)

{'t0': np.float64(0.4999999999991462), 't1': np.float64(1.2932038153935697)}


In [52]:
# ── guides/gradients.mdx #5 ──
from superfermion.qml.gradient.riemannian import riemannian_gradient
import numpy as np

state = sf.simulate(ansatz, params=params)
dag = ansatz.to_ir()
metric = state.qfim(dag, params)                       # QFIM as metric tensor

grad = adjoint_grad_vector(ansatz, H, ["t0", "t1"], np.array([params["t0"], params["t1"]]))

natural_grad = riemannian_gradient(grad, metric)
print(natural_grad)

[ 8.53836373e-12 -9.32039086e-01]


In [53]:
# ── guides/gradients.mdx #6 ──
import superfermion as sf
import numpy as np

H = sf.Hamiltonian([
    sf.PauliString("II", coeff=-1.0523),
    sf.PauliString("IZ", coeff=0.3979),
    sf.PauliString("ZI", coeff=-0.3979),
    sf.PauliString("ZZ", coeff=-0.0112),
    sf.PauliString("XX", coeff=0.1809),
])

ansatz = (sf.Circuit(2)
    .ry(sf.param("t0"), 0).ry(sf.param("t1"), 1)
    .cnot(0, 1)
    .ry(sf.param("t2"), 0).ry(sf.param("t3"), 1))

params = {f"t{i}": np.random.uniform(0, np.pi) for i in range(4)}
obs = [([0, 0], -1.0523, 0.0),   # II
       ([0, 3], 0.3979, 0.0),    # IZ
       ([3, 0], -0.3979, 0.0),   # ZI
       ([3, 3], -0.0112, 0.0),   # ZZ
       ([1, 1], 0.1809, 0.0)]    # XX
lr = 0.1

for step in range(100):
    state = sf.simulate(ansatz, params=params)
    energy = state.expectation(obs)

    dag = ansatz.bind(params).to_ir()
    grads = state.grad(obs, dag, params)

    for k in params:
        params[k] -= lr * grads.get(k, 0.0)

    if step % 20 == 0:
        print(f"Step {step:3d}: energy = {energy:.8f}")

print(f"Final energy: {energy:.8f}")

Step   0: energy = -0.82241399
Step  20: energy = -0.82241399
Step  40: energy = -0.82241399
Step  60: energy = -0.82241399
Step  80: energy = -0.82241399
Final energy: -0.82241399


In [54]:
# ── guides/gradients.mdx #7 ──
from superfermion.algorithms.variational import VQE

vqe = VQE(ansatz, H)
result = vqe.minimize(iterations=100)
print(f"Energy: {result.optimal_value:.8f}")
print(f"Optimal params: {result.optimal_params}")

Energy: -1.85720198
Optimal params: {'t0': 0.3563618410232574, 't1': -0.8822853770515164, 't2': 3.4215053473895836, 't3': 0.9138920099788675}


In [55]:
# ── guides/gradients.mdx #8 ──
from superfermion.algorithms.variational import QAOA
import networkx as nx

# MaxCut on a random 8-node graph
G = nx.random_regular_graph(3, 8)

qaoa = QAOA(n_qubits=8, edges=list(G.edges()), p_layers=2)
result = qaoa.minimize(iterations=100)
print(f"MaxCut value: {result.metadata['max_cut_value']}")
print(f"Best bitstring: {result.metadata['best_bitstring']}")

ModuleNotFoundError: No module named 'networkx'

In [ ]:
# ── guides/gradients.mdx #9 ──
from scipy.optimize import minimize

def cost_function(param_array):
    params = {f"t{i}": v for i, v in enumerate(param_array)}
    state = sf.simulate(ansatz, params=params)
    return state.expectation(obs)

result = minimize(cost_function, x0=np.zeros(4), method="L-BFGS-B")
print(f"Optimal: {result.fun:.8f} at {result.x}")

Optimal: -1.24440000 at [-1.57079635  0.          0.          0.        ]


In [ ]:
# ── guides/gradients.mdx #10 ──
import numpy as np

state = sf.simulate(ansatz, params=params)
dag = ansatz.to_ir()               # unbound DAG — needed for QFIM
qfim = state.qfim(dag, params)     # n_params × n_params matrix

print(qfim.shape)
print(f"Condition number: {np.linalg.cond(qfim):.2f}")

(4, 4)
Condition number: inf


## Page: `guides/chemistry.mdx`

6 Python block(s) — run top-to-bottom.

In [ ]:
# ── guides/chemistry.mdx #0 ──
from superfermion.chemistry.hamiltonians import get_molecular_hamiltonian
from superfermion.chemistry.ansatz import uccsd_ansatz
import superfermion as sf

# Get a pre-built molecular Hamiltonian
H = get_molecular_hamiltonian("H2", basis="sto-3g")
print(f"H2 Hamiltonian: {len(H.terms)} terms")

# Build a UCCSD ansatz (minimal 2-qubit H2 ansatz)
ansatz = uccsd_ansatz(n_qubits=2, n_electrons=2)
print(f"Ansatz: {ansatz.n_parameters} parameter, {ansatz.gate_count} gates")

# Run VQE
from superfermion.algorithms.variational import VQE
vqe = VQE(ansatz, H)
result = vqe.minimize(iterations=100)
print(f"H2 ground state energy: {result.optimal_value:.6f} Hartree")

H2 Hamiltonian: 5 terms
Ansatz: 1 parameter, 3 gates
H2 ground state energy: -1.137306 Hartree


In [ ]:
# ── guides/chemistry.mdx #1 ──
from superfermion.chemistry.hamiltonians import get_molecular_hamiltonian

# Built-in molecules (pre-computed at equilibrium geometry)
h2 = get_molecular_hamiltonian("H2", basis="sto-3g")
print(f"H2: {len(h2.terms)} terms")

# Additional molecules (LiH, H2O, ...) require PySCF — install with:
#   pip install "superfermion[chemistry]"

H2: 5 terms


In [ ]:
# ── guides/chemistry.mdx #2 ──
from superfermion.chemistry.pyscf_bridge import molecule_from_geometry

# Define any molecule via an XYZ geometry string
mol = molecule_from_geometry("H 0 0 0; H 0 0 0.7414", basis="sto-3g", name="H2")

# With PySCF installed this computes HF + integrals; otherwise it falls
# back to the pre-computed molecule library
summary = mol.summary()
print(f"{summary['name']}: {summary['n_electrons']} electrons, "
      f"{summary['n_orbitals']} orbitals, {summary['n_qubits']} qubits, "
      f"pyscf={'available' if summary['pyscf_available'] else 'library fallback'}")

# Build the qubit Hamiltonian
H = mol.to_hamiltonian(active_space=(2, 2))
print(f"Qubit Hamiltonian: {len(H.terms)} terms")

H2: 2 electrons, 2 orbitals, 4 qubits, pyscf=library fallback
Qubit Hamiltonian: 6 terms


In [ ]:
# ── guides/chemistry.mdx #3 ──
from superfermion.chemistry.hamiltonians import FermionicOperator
import numpy as np

# One-body integrals h1 (spin-orbital basis) — a two-site model
h1 = np.zeros((2, 2))
h1[0, 0] = -1.0
h1[1, 1] = -0.5
h1[0, 1] = h1[1, 0] = 0.2

fermion_op = FermionicOperator.from_coeffs(h1)

# Jordan-Wigner transform
jw_qubit_op = fermion_op.jordan_wigner(n_qubits=2)
print(f"Jordan-Wigner: {len(jw_qubit_op.terms)} terms")

# Bravyi-Kitaev transform
bk_qubit_op = fermion_op.bravyi_kitaev(n_qubits=2)
print(f"Bravyi-Kitaev: {len(bk_qubit_op.terms)} terms")

Jordan-Wigner: 5 terms
Bravyi-Kitaev: 6 terms


In [ ]:
# ── guides/chemistry.mdx #4 ──
from superfermion.chemistry.ansatz import uccsd_ansatz

# Minimal 2-qubit H2 ansatz (single parameter, O'Malley et al. 2016)
h2_ansatz = uccsd_ansatz(n_qubits=2, n_electrons=2)
print(f"H2 ansatz: {h2_ansatz.n_parameters} parameter, {h2_ansatz.gate_count} gates")

# Hardware-efficient UCC-style ansatz for larger active spaces
uccsd = uccsd_ansatz(n_qubits=4, n_electrons=2)
print(f"4-qubit ansatz: {uccsd.n_parameters} parameters, {uccsd.gate_count} gates")

H2 ansatz: 1 parameter, 3 gates
4-qubit ansatz: 5 parameters, 13 gates


In [ ]:
# ── guides/chemistry.mdx #5 ──
from superfermion.chemistry.hamiltonians import get_molecular_hamiltonian
from superfermion.chemistry.ansatz import uccsd_ansatz
from superfermion.algorithms.variational import VQE
import numpy as np

# 1. Build molecular Hamiltonian (pre-computed H2 library entry)
H = get_molecular_hamiltonian("H2", basis="sto-3g")

# 2. Build ansatz
ansatz = uccsd_ansatz(2, 2)

# 3. Initial parameters (array, ordered as ansatz.parameters)
init_params = np.zeros(ansatz.n_parameters)

# 4. Run VQE
vqe = VQE(ansatz, H)
result = vqe.minimize(initial_params=init_params, iterations=200)
print(f"Ground state energy: {result.optimal_value:.8f} Hartree")
print(f"Optimal parameters: {result.optimal_params}")

Ground state energy: -1.13730604 Hartree
Optimal parameters: {'theta': -2.9180556414201315}


## Page: `guides/qec.mdx`

4 Python block(s) — run top-to-bottom.

In [ ]:
# ── guides/qec.mdx #0 ──
from superfermion.qec import (
    RepetitionCode,
    ShorCode,
    SteaneCode,
    BaconShorCode,
    SurfaceCode2D,
    ToricCode2D,
    ColorCode,
    HoneycombCode,
    HypercubeCode4D,
    GenericCSSCode,
)

In [ ]:
# ── guides/qec.mdx #1 ──
import numpy as np
from superfermion.qec import SurfaceCode2D, MWPMDecoder

# Create a distance-3 surface code
code = SurfaceCode2D(distance=3)
print(f"Code parameters: data={code.n_data}, ancilla={code.n_ancilla}, distance={code.d}")

# Build the syndrome-extraction circuit (data + ancilla qubits)
syndrome_circuit = code.build()
print(f"Syndrome circuit qubits: {syndrome_circuit.n_qubits}")

# Decode errors: decoders need the data-qubit count and the map of which
# data qubits each syndrome (ancilla) qubit touches
syndrome_qubit_map = [[i, (i + 1) % code.n_data] for i in range(code.n_ancilla)]
decoder = MWPMDecoder(code.n_data, syndrome_qubit_map)

syndrome = np.zeros(code.n_ancilla, dtype=int)   # example syndrome (no error)
correction = decoder.decode(syndrome)
print(f"Correction: {correction}")

Code parameters: data=9, ancilla=8, distance=3
Syndrome circuit qubits: 17
Correction: []


In [ ]:
# ── guides/qec.mdx #2 ──
import numpy as np
from superfermion.qec import SurfaceCode2D, MWPMDecoder, UnionFindDecoder

code = SurfaceCode2D(distance=5)
syndrome_qubit_map = [[i, (i + 1) % code.n_data] for i in range(code.n_ancilla)]

# MWPM — accurate, moderate speed
mwpm = MWPMDecoder(code.n_data, syndrome_qubit_map)

# Union-Find — fast, slightly lower threshold
uf = UnionFindDecoder(code.n_data, syndrome_qubit_map)

# Both return a list of (qubit_index, pauli_type) corrections
syndrome = np.zeros(code.n_ancilla, dtype=int)
syndrome[[0, 3]] = 1                    # example syndrome bits
mwpm_correction = mwpm.decode(syndrome)
uf_correction = uf.decode(syndrome)
print(f"MWPM correction: {mwpm_correction}")
print(f"UF correction:   {uf_correction}")

MWPM correction: [(1, 'X'), (2, 'X'), (3, 'X')]
UF correction:   [(1, 'X'), (2, 'X'), (3, 'X')]


In [ ]:
# ── guides/qec.mdx #3 ──
from superfermion.qec import QECManager

manager = QECManager()

# Simulate the full logical-qubit lifecycle: encode → noise → syndrome → recovery
result = manager.run_logical_lifecycle("surface_2d")
print(f"Phase: {result['phase']}")
print(f"Logical state: {result['logical_state']}")
print(f"Syndrome detected: {result['syndrome_detected']}")
print(f"Error corrected: {result['error_corrected']}")

# Simulate a fault-tolerant workflow
workflow = manager.simulate_fault_tolerant_workflow("surface_2d")
print(f"Code: {workflow['code']}, qubits: {workflow['qubits']}")
print(f"Fidelity estimate: {workflow['fidelity_estimate']}")

Phase: Recovery Complete
Logical state: Protected
Syndrome detected: 10110110000000001
Error corrected: True
Code: surface_2d, qubits: 17
Fidelity estimate: 0.999


## Page: `guides/interop.mdx`

9 Python block(s) — run top-to-bottom.

In [ ]:
# ── guides/interop.mdx #0 ──
from superfermion.bridge import from_qiskit, to_qiskit
from qiskit import QuantumCircuit
import superfermion as sf

# Qiskit → Superfermion
qiskit_qc = QuantumCircuit(2)
qiskit_qc.h(0)
qiskit_qc.cx(0, 1)
sf_qc = from_qiskit(qiskit_qc)

# Superfermion → Qiskit
sf_qc = sf.Circuit(2).h(0).cnot(0, 1)
qiskit_qc = to_qiskit(sf_qc)

# Run on Qiskit Aer from Superfermion
from qiskit_aer import AerSimulator
qiskit_qc.measure_all()   # measurements so get_counts() has results
result = AerSimulator().run(qiskit_qc).result()
print(result.get_counts())

{'11 00': 501, '00 00': 523}

In [ ]:
# ── guides/interop.mdx #1 ──
from superfermion.bridge import from_cirq, to_cirq

# Cirq → Superfermion
import cirq
cirq_qc = cirq.Circuit(cirq.H(cirq.LineQubit(0)), cirq.CNOT(cirq.LineQubit(0), cirq.LineQubit(1)))
sf_qc = from_cirq(cirq_qc)

# Superfermion → Cirq
cirq_qc = to_cirq(sf_qc)

In [ ]:
# ── guides/interop.mdx #2 ──
from superfermion.bridge import from_pennylane, to_pennylane
import pennylane as qml
import superfermion as sf

# PennyLane → Superfermion
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def circuit(x):
    qml.RY(x, wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()

# Extract the tape and convert
pl_tape = qml.workflow.construct_tape(circuit)(0.5)
sf_qc = from_pennylane(pl_tape)
print(sf.run(sf_qc.bind({"x": 0.5}), shots=100).counts)

# Superfermion → PennyLane (returns a quantum function for a QNode)
qfunc = to_pennylane(sf.Circuit(2).h(0).cnot(0, 1))

@qml.qnode(dev)
def pl_circuit():
    qfunc()
    return qml.state()

print(len(pl_circuit()))  # 4 amplitudes

{'00': 97, '11': 3}
4


In [ ]:
# ── guides/interop.mdx #3 ──
from superfermion.bridge import from_qasm, to_qasm

# Parse QASM (Rust-accelerated parser)
qasm_str = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
h q[0];
cx q[0], q[1];
"""
sf_qc = from_qasm(qasm_str)

# Export to QASM 2
qasm_output = to_qasm(sf_qc)

In [ ]:
# ── guides/interop.mdx #4 ──
# Export (built into Circuit)
qasm3_str = sf.Circuit(2).h(0).cnot(0, 1).to_qasm3()

# Import
from superfermion.serialization import from_qasm3
sf_qc = from_qasm3(qasm3_str)

In [ ]:
# ── guides/interop.mdx #5 ──
from superfermion.bridge import to_braket
import superfermion as sf

# Export Superfermion circuit to AWS Braket format
sf_qc = sf.Circuit(2).h(0).cnot(0, 1)
braket_circuit = to_braket(sf_qc)
print(type(braket_circuit).__name__)  # braket.circuits.Circuit

# Submit to AWS Braket (requires AWS credentials)
# from braket.aws import AwsDevice
# device = AwsDevice("arn:aws:braket:::device/quantum-simulator/amazon/sv1")
# device.run(braket_circuit, shots=1000)

Circuit


In [ ]:
# ── guides/interop.mdx #6 ──
from superfermion.bridge import to_ionq

# Export to IonQ's native format
ionq_circuit = to_ionq(sf_qc)

In [ ]:
# ── guides/interop.mdx #7 ──
# Serialize
json_str = sf.Circuit(2).h(0).cnot(0, 1).to_json()

# Deserialize
restored = sf.Circuit.from_json(json_str)
assert restored.gate_count == 2

In [ ]:
# ── guides/interop.mdx #8 ──
gates = sf.Circuit(2).h(0).cnot(0, 1).to_gate_list()
# [{"name": "H", "qubits": [0]}, {"name": "CNOT", "qubits": [0, 1]}]

# Rebuild from gate list (names are uppercase, lowercase the method name)
qc = sf.Circuit(2)
for g in gates:
    qc = getattr(qc, g["name"].lower())(*g.get("params", []), *g["qubits"])
print(qc.gate_count)  # 2

2


## Page: `guides/ml-frameworks.mdx`

5 Python block(s) — run top-to-bottom.

In [ ]:
# ── guides/ml-frameworks.mdx #1 ──
from superfermion.nn.quantum_layer import QuantumLayer
import jax
import jax.numpy as jnp
import superfermion as sf

# Build a parameterized circuit
theta = sf.param("theta")
phi = sf.param("phi")
circuit = sf.Circuit(2).ry(theta, 0).ry(phi, 1).cnot(0, 1).rz(theta, 1)

# Define observable (pass a PauliString — Flax converts plain lists to tuples)
obs = sf.PauliString("ZZ", coeff=1.0)

# Create layer
layer = QuantumLayer(circuit, obs, device="cpu")

# Initialize
key = jax.random.PRNGKey(0)
params = layer.init(key)

# Forward pass (custom_vjp routes backward to sf.State.grad)
output = layer.apply(params)
print(f"Output: {output}")

# Gradient via JAX autograd
grad_fn = jax.grad(lambda p: layer.apply(p).sum())
grads = grad_fn(params)
print(f"Gradients: {grads}")

Output: 0.9449097514152527
Gradients: {'params': {'weights': Array([-1.4311469e-17, -3.2733098e-01], dtype=float32)}}


In [ ]:
# ── guides/ml-frameworks.mdx #2 ──
from superfermion.nn.torch_layer import TorchQuantumLayer
import torch
import superfermion as sf

# Build circuit and observable
circuit = sf.Circuit(2).ry(sf.param("t0"), 0).ry(sf.param("t1"), 1).cnot(0, 1)
obs = [([3, 3], 1.0, 0.0)]  # ZZ

# Create layer — parameters live inside the module (layer.weights)
layer = TorchQuantumLayer(circuit, obs, device="cpu")

# Forward pass
output = layer()
print(f"Output: {output.item():.6f}")

# Backward pass (sf.State.grad via autograd.Function)
output.backward()
print(f"Gradients: {layer.weights.grad}")

Output: 0.370735
Gradients: tensor([ 0.0000, -0.9287], dtype=torch.float64)


In [ ]:
# ── guides/ml-frameworks.mdx #3 ──
from superfermion.nn.tf_layer import TFQuantumLayer
import tensorflow as tf
import superfermion as sf

# Build circuit and observable
circuit = sf.Circuit(2).ry(sf.param("t0"), 0).ry(sf.param("t1"), 1).cnot(0, 1)
obs = [([3, 3], 1.0, 0.0)]  # ZZ

# Create layer — weights live inside the layer (layer.weights_var)
layer = TFQuantumLayer(circuit, obs, device="cpu")

# Forward pass
output = layer()
print(f"Output: {output.numpy():.6f}")

# Backward pass (tf.custom_gradient routes to sf.State.grad)
with tf.GradientTape() as tape:
    loss = layer()

grads = tape.gradient(loss, layer.weights_var)
print(f"Gradients: {grads.numpy()}")

Output: 0.114653
Gradients: [ 0.        -1.9868113]


In [ ]:
# ── guides/ml-frameworks.mdx #4 ──
import jax
import torch
import superfermion as sf
from superfermion.nn.quantum_layer import QuantumLayer
from superfermion.nn.torch_layer import TorchQuantumLayer

circuit = sf.Circuit(2).ry(sf.param("t0"), 0).ry(sf.param("t1"), 1).cnot(0, 1)
obs = sf.PauliString("ZZ", coeff=1.0)

# JAX/Flax: evaluate a batch of weight vectors
flax_layer = QuantumLayer(circuit, obs, device="cpu")
params = flax_layer.init(jax.random.PRNGKey(0))
weights_batch = [params["params"]["weights"], params["params"]["weights"] * 2]
outputs = [flax_layer.apply({"params": {"weights": w}}) for w in weights_batch]
print(f"JAX batch outputs: {outputs}")

# PyTorch: stack repeated evaluations (weights live inside the module)
torch_layer = TorchQuantumLayer(circuit, obs, device="cpu")
outputs = torch.stack([torch_layer() for _ in range(4)])
print(f"PyTorch batch outputs: {outputs}")

JAX batch outputs: [Array(0.94490975, dtype=float32, weak_type=True), Array(0.78570884, dtype=float32, weak_type=True)]
PyTorch batch outputs: tensor([-0.1561, -0.1561, -0.1561, -0.1561], dtype=torch.float64,
       grad_fn=<StackBackward0>)


In [ ]:
# ── guides/ml-frameworks.mdx #5 ──
import superfermion as sf
import numpy as np

ansatz = sf.Circuit(2).ry(sf.param("t0"), 0).ry(sf.param("t1"), 1).cnot(0, 1)
obs = [([3, 3], 1.0, 0.0)]

params = {"t0": 0.5, "t1": 1.2}
lr = 0.1

for step in range(100):
    state = sf.simulate(ansatz, params=params)
    energy = state.expectation(obs)

    dag = ansatz.bind(params).to_ir()
    grads = state.grad(obs, dag, params)

    for k in params:
        params[k] -= lr * grads.get(k, 0.0)

    if step % 20 == 0:
        print(f"Step {step}: energy = {energy:.6f}")

Step 0: energy = 0.362358
Step 20: energy = 0.362358
Step 40: energy = 0.362358
Step 60: energy = 0.362358
Step 80: energy = 0.362358


## Page: `guides/providers.mdx`

8 Python block(s) — run top-to-bottom.

In [ ]:
# ── guides/providers.mdx #2 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# from superfermion.devices.ibm import IBMDevice
# import superfermion as sf
#
# # Connect
# ibm = IBMDevice(token="your-ibm-token")
#
# # List available devices
# print(ibm.list_devices())
#
# # Run on real hardware
# qc = sf.Circuit(2).h(0).cnot(0, 1).measure_all()
# result = sf.run(qc, device=ibm("ibm_brisbane"), shots=8192)
#
# print(result.counts)
# print(result.metadata.execution_time)
# print(result.metadata.device_properties)
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #3 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# # With error mitigation
# from superfermion.mitigation import readout_correction, zne
#
# # Readout error correction
# result = sf.run(qc, device=ibm("ibm_brisbane"), shots=8192)
# corrected_counts = readout_correction(result)
#
# # Zero-noise extrapolation
# result = zne(qc, device=ibm("ibm_brisbane"), noise_levels=[1, 2, 3])
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #5 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# from superfermion.devices.ionq import IonQDevice
# import superfermion as sf
#
# # Connect
# ionq = IonQDevice(api_key="your-ionq-key")
#
# # Run on trapped-ion hardware
# qc = sf.Circuit(3).h(0).cnot(0, 1).cnot(1, 2)
# result = sf.run(qc, device=ionq("aria-1"), shots=1000)
#
# print(result.counts)
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #7 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# from superfermion.devices.braket import BraketDevice
# import superfermion as sf
#
# # Connect (uses AWS credentials from environment or ~/.aws)
# braket = BraketDevice(s3_bucket="my-braket-results")
#
# # Run on AWS hardware
# qc = sf.Circuit(2).h(0).cnot(0, 1).measure_all()
# result = sf.run(qc, device=braket("sv1"), shots=1000)  # simulator
# # result = sf.run(qc, device=braket("rigetti_aspen_m3"), shots=1000)  # hardware
#
# print(result.counts)
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #8 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# # List available devices
# devices = braket.list_devices()
# for d in devices:
#     print(f"{d.name}: {d.n_qubits} qubits, {d.status}")
#
# # Simulators
# result = sf.run(qc, device=braket("sv1"))    # statevector simulator
# result = sf.run(qc, device=braket("tn1"))    # tensor network simulator
# result = sf.run(qc, device=braket("dm1"))    # density matrix simulator
#
# # Hardware (requires appropriate AWS region access)
# # braket("rigetti_aspen_m3")   # Rigetti
# # braket("ionq_harmony")       # IonQ via Braket
# # braket("oqc_lucy")           # Oxford Quantum Circuits
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #9 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# from superfermion.devices.openquantum import OpenQuantumDevice
# import superfermion as sf
#
# # Connect to any OpenQuantum-compatible device
# oq = OpenQuantumDevice(endpoint="https://api.example.com", api_key="...")
# result = sf.run(qc, device=oq("device_01"), shots=1000)
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── guides/providers.mdx #10 ──
# Local baseline (runs here)
qc = sf.Circuit(2).h(0).cnot(0, 1).measure_all()
with sf.experiment("qpu-benchmark") as tracker:
    sf.run(qc, device="cpu", shots=10000)

    # QPU runs — skipped in this notebook (need real credentials):
    # sf.run(qc, device=ibm("ibm_brisbane"), shots=8192)
    # sf.run(qc, device=ionq("aria-1"), shots=1000)

# All runs logged in ~/.superfermion/runs/qpu-benchmark/
print(tracker.runs)

[{'device': 'cpu', 'shots': 10000, 'n_qubits': 2, 'depth': 3, 'gate_count': 4, 'started_at': 1786738763.682331, 'completed_at': 1786738763.683926, 'n_outcomes': 2}]


In [ ]:
# ── guides/providers.mdx #11 ──
import superfermion as sf
from superfermion.devices import DeviceExecutor, DeviceCapabilities

class MyCustomDevice:
    """Any class with execute() + capabilities() satisfies the protocol."""
    def execute(self, circuit, shots, **kwargs):
        # Simulate the submitted circuit locally and return a RunResult
        return sf.run(circuit, device="cpu", shots=shots)

    def capabilities(self):
        return DeviceCapabilities()

# Use directly — no registration needed
qc = sf.Circuit(2).h(0).cnot(0, 1)
my_device = MyCustomDevice()
assert isinstance(my_device, DeviceExecutor)   # structural check
result = sf.run(qc, device=my_device, shots=1000)
print(result.counts)

{'00': 492, '11': 508}


## Page: `index.mdx`

1 Python block(s) — run top-to-bottom.

In [ ]:
# ── index.mdx #1 ──
import superfermion as sf

# Bell state — 3 lines
qc = sf.Circuit(2).h(0).cx(0, 1)
result = sf.run(qc, shots=1024)
print(result.counts)  # {'00': ~512, '11': ~512}

# Gradient — exact, O(1) scaling
ansatz = sf.Circuit(2).ry(sf.param("t"), 0).cnot(0, 1).ry(sf.param("p"), 1)
state = sf.simulate(ansatz, params={"t": 0.5, "p": 1.2})
obs = [([3, 3], 1.0, 0.0)]      # Pauli ZZ (Rust format: 0=I 1=X 2=Y 3=Z)
grads = state.grad(obs, ansatz.bind({"t": 0.5, "p": 1.2}).to_ir(), {"t": 0.5, "p": 1.2})
print(grads)  # {'t': ..., 'p': ...}

{'11': 516, '00': 508}
{}


## Page: `reference/api.mdx`

24 Python block(s) — run top-to-bottom.

In [ ]:
# ── reference/api.mdx #1 ──
import sys
sys.stdout.reconfigure(encoding="utf-8", errors="replace")
import superfermion as sf

qc = sf.Circuit(2).h(0).cnot(0, 1)

print(qc.n_qubits)     # 2
print(qc.gate_count)   # 2
print(qc.depth)        # 2
print(qc.draw())       # ASCII diagram

2
2
2
q0: ─  [H]  ─   ●   ─
q1: ─  ───  ─   ⊕   ─


In [ ]:
# ── reference/api.mdx #2 ──
import superfermion as sf

qc = sf.Circuit(2).h(0).cnot(0, 1)

result = sf.run(qc, shots=4096)
print(result.counts)        # {'00': 2048, '11': 2048}
print(result.state)         # sf.State (Rust handle)
print(result.shots)         # 4096

{'00': 2005, '11': 2091}
State(n_qubits=2, method='statevector', device='cpu')
4096


In [ ]:
# ── reference/api.mdx #3 ──
# Local simulation (builtin shorthands)
sf.run(qc, device="cpu")                  # Rust CPU (Rayon + AVX)
# sf.run(qc, device="gpu")                # Rust GPU (CUDA); skipped: no GPU here

# QPU via provider objects — skipped in this notebook (need real credentials):
# from superfermion.devices.ibm import IBMDevice
# ibm = IBMDevice(token="...")
# sf.run(qc, device=ibm("ibm_brisbane"))    # explicit object, not magic string
# from superfermion.devices.ionq import IonQDevice
# ionq = IonQDevice(api_key="...")
# sf.run(qc, device=ionq("aria-1"))         # same pattern, different provider
print(list(sf.run(qc, device="cpu", shots=128).counts.items()))

[('11', 62), ('00', 66)]


In [ ]:
# ── reference/api.mdx #4 ──
state = sf.simulate(qc)

# sf.State is Rust-native — all methods dispatch to Rust
print(state.n_qubits)           # 2
print(state.entropy())          # Von Neumann entropy
print(state.purity())           # state purity
print(state.fidelity(state))    # 1.0

sv = state.numpy()              # export to numpy
samples = state.sample(1000)    # fast Rust sampling

2
0.0
1.0
0.9999999999999996


In [ ]:
# ── reference/api.mdx #5 ──
result = sf.run(qc, method="statevector", shots=1000)

In [ ]:
# ── reference/api.mdx #6 ──
result = sf.run(qc, method="mps", bond_dim=64, shots=1000)

In [ ]:
# ── reference/api.mdx #7 ──
clifford = sf.Circuit(100).h(0)
for i in range(99):
    clifford = clifford.cnot(i, i + 1)

result = sf.run(clifford, method="stabilizer", shots=1000)

In [ ]:
# ── reference/api.mdx #8 ──
noise = sf.NoiseModel().add_depolarizing(0.01)
result = sf.run(qc, method="density_matrix", noise_model=noise, shots=0)

In [ ]:
# ── reference/api.mdx #9 ──
import superfermion as sf
from superfermion.compiler.specs import HardwareSpec

qc = sf.Circuit(5).h(0).cnot(0, 3).cnot(1, 4).swap(2, 3)

# Optimization only (no hardware target)
optimized = sf.compile(qc, level=1)

# Compile for a specific topology and gate set
target = HardwareSpec(
    name="my_device",
    n_qubits=5,
    native_gates=["rz", "sx", "x", "cx"],
    coupling_map=[(0,1), (1,2), (2,3), (3,4)],
)
compiled = sf.compile(qc, level=2, target=target)

In [ ]:
# ── reference/api.mdx #10 ──
from superfermion.compiler import (
    PassManager,
    BasisTranslationPass,
    PauliTwirlingPass,
    DynamicalDecouplingPass,
    UnitaryDecompositionPass,
    SchedulingPass,
    apply_dynamical_decoupling,
    apply_noise_suppression,
    schedule_circuit,
)

In [ ]:
# ── reference/api.mdx #11 ──
import superfermion as sf

# Build observables
H = sf.Hamiltonian([
    sf.PauliString("ZZ", coeff=0.5),
    sf.PauliString("XI", coeff=0.3),
    sf.PauliString("IZ", coeff=0.2),
])

# Exact expectation from state
state = sf.simulate(sf.Circuit(2).h(0).cnot(0, 1))
energy = H.expectation(state.numpy())
print(f"Exact <H> = {energy:.6f}")

# Estimate from measurement counts
result = sf.run(sf.Circuit(2).h(0).cnot(0, 1), shots=10000)
obs = [([3, 3], 0.5, 0.0),   # ZZ
       ([1, 0], 0.3, 0.0),   # XI
       ([0, 3], 0.2, 0.0)]   # IZ
energy = result.expectation(obs)
print(f"Counts <H> = {energy:.6f}")

Exact <H> = 0.500000
Counts <H> = 0.500000


In [ ]:
# ── reference/api.mdx #12 ──
# PauliString
p = sf.PauliString("XZ", coeff=0.5)
print(p.pauli_str)   # "XZ"
print(p.coeffs)      # 0.5

# Hamiltonian
H = sf.Hamiltonian([p, sf.PauliString("II", coeff=-1.0)])
print(len(H.terms))        # 2
sparse = H.to_sparse_pauli_op()

# SparsePauliOp
op = sf.SparsePauliOp.from_dict({"ZZ": 0.5, "XI": 0.3})

# Convenience Pauli matrices
from superfermion.observables import I, X, Y, Z

# expval shortcut
from superfermion import expval
state = sf.simulate(sf.Circuit(2).h(0).cnot(0, 1))
val = expval(state.numpy(), H)
print(f"<H> = {val:.6f}")

XZ
0.5
2
<H> = -1.000000


In [ ]:
# ── reference/api.mdx #13 ──
# List of (pauli_indices, coeff_real, coeff_imag)
# Pauli encoding: 0=I, 1=X, 2=Y, 3=Z
zz_obs = [([3, 3], 1.0, 0.0)]     # ZZ with coefficient 1.0
xz_obs = [([1, 3], 0.5, 0.0)]     # 0.5 * XZ

In [ ]:
# ── reference/api.mdx #14 ──
import superfermion as sf

theta = sf.param("theta")
phi = sf.param("phi")

ansatz = (sf.Circuit(2)
    .ry(theta, 0)
    .ry(phi, 1)
    .cnot(0, 1)
    .rz(theta, 1))

print(ansatz.n_parameters)   # 2
print(ansatz.parameters)     # ['theta', 'phi']

# Bind and compute gradients
params = {"theta": 0.5, "phi": 1.2}
state = sf.simulate(ansatz, params=params)

obs = [([3, 3], 1.0, 0.0)]  # ZZ
dag = ansatz.bind(params).to_ir()
grads = state.grad(obs, dag, params)
print(grads)  # {"theta": -0.23..., "phi": 0.41...}

2
['theta', 'phi']
{}


In [ ]:
# ── reference/api.mdx #15 ──
import superfermion as sf
import numpy as np

H = sf.Hamiltonian([
    sf.PauliString("II", coeff=-1.0523),
    sf.PauliString("IZ", coeff=0.3979),
    sf.PauliString("ZI", coeff=-0.3979),
    sf.PauliString("ZZ", coeff=-0.0112),
    sf.PauliString("XX", coeff=0.1809),
])

ansatz = (sf.Circuit(2)
    .ry(sf.param("t0"), 0)
    .ry(sf.param("t1"), 1)
    .cnot(0, 1)
    .ry(sf.param("t2"), 0)
    .ry(sf.param("t3"), 1))

params = {f"t{i}": np.random.uniform(0, np.pi) for i in range(4)}
obs = [([0, 0], -1.0523, 0.0),   # II
       ([0, 3], 0.3979, 0.0),    # IZ
       ([3, 0], -0.3979, 0.0),   # ZI
       ([3, 3], -0.0112, 0.0),   # ZZ
       ([1, 1], 0.1809, 0.0)]    # XX
lr = 0.1

for step in range(50):
    state = sf.simulate(ansatz, params=params)
    energy = state.expectation(obs)

    dag = ansatz.bind(params).to_ir()
    grads = state.grad(obs, dag, params)

    for key in params:
        params[key] -= lr * grads.get(key, 0.0)

    if step % 10 == 0:
        print(f"Step {step}: energy = {energy:.6f}")

Step 0: energy = -1.078681
Step 10: energy = -1.078681
Step 20: energy = -1.078681
Step 30: energy = -1.078681
Step 40: energy = -1.078681


In [ ]:
# ── reference/api.mdx #16 ──
import superfermion as sf
import numpy as np
from superfermion.algorithms.variational import VQE

H = sf.Hamiltonian([
    sf.PauliString("II", coeff=-1.0523),
    sf.PauliString("IZ", coeff=0.3979),
    sf.PauliString("ZI", coeff=-0.3979),
    sf.PauliString("ZZ", coeff=-0.0112),
    sf.PauliString("XX", coeff=0.1809),
])

ansatz = (sf.Circuit(2)
    .ry(sf.param("t0"), 0)
    .ry(sf.param("t1"), 1)
    .cnot(0, 1)
    .ry(sf.param("t2"), 0)
    .ry(sf.param("t3"), 1))

# Use the built-in VQE class
vqe = VQE(ansatz, H)
result = vqe.minimize(iterations=100)
print(f"Energy: {result.optimal_value:.8f}")
print(f"Optimal params: {result.optimal_params}")

Energy: -1.85720198
Optimal params: {'t0': 0.3563618410232575, 't1': -0.8822853770515168, 't2': 3.4215053473895836, 't3': 0.9138920099788678}


In [ ]:
# ── reference/api.mdx #17 ──
from superfermion.pulse import (
    Schedule, GaussianPulse, DRAGPulse, SquarePulse,
    GaussianSquarePulse, CosinePulse, Waveform,
    Channel, ChannelType, CalibrationSet, GateCalibration,
)

# Build a pulse schedule
s = Schedule()

# Add a Gaussian pulse on the drive channel (duration/sigma in dt units)
gaussian = GaussianPulse(duration=64, sigma=16, amp=0.5)
s.add(gaussian, channel="d0", t0=0)

# Add a DRAG pulse for reduced leakage
drag = DRAGPulse(duration=64, sigma=16, amp=0.5, beta=0.1)
s.add(drag, channel="d0", t0=100)

# Add a square pulse for measurement
square = SquarePulse(duration=200, amp=1.0)
s.add(square, channel="m0", t0=200)
print(s)

# Compile a circuit to a schedule
from superfermion.compiler import schedule_circuit
qc = sf.Circuit(1).rx(0.5, 0)
scheduled = schedule_circuit(qc)
print(scheduled)

Schedule(instructions=3, duration=400dt, channels=['d0', 'm0'])
Schedule(gates=[ScheduledGate(gate=GateRecord(name='RX', qubits=[0], params=[0.5], classical_bits=[]), start_time=np.float64(0.0), duration=1.0, qubits=[0])], total_duration=np.float64(1.0), n_qubits=1, critical_path=['RX'])


In [ ]:
# ── reference/api.mdx #18 ──
import numpy as np
from superfermion.qec import (
    SurfaceCode2D, SteaneCode, RepetitionCode,
    MWPMDecoder, UnionFindDecoder, BPOSD_Decoder, NeuralDecoder,
    QECManager,
)

# Create a surface code
code = SurfaceCode2D(distance=3)
syndrome_qubit_map = [[i, (i + 1) % code.n_data] for i in range(code.n_ancilla)]

# Decode errors
decoder = MWPMDecoder(code.n_data, syndrome_qubit_map)
correction = decoder.decode(np.zeros(code.n_ancilla, dtype=int))
print(f"Correction: {correction}")

# Full QEC pipeline
manager = QECManager()
result = manager.run_logical_lifecycle("surface_2d")
print(f"Logical state: {result['logical_state']}")

Correction: []
Logical state: Protected


In [ ]:
# ── reference/api.mdx #19 ──
import numpy as np
from superfermion.chemistry.hamiltonians import get_molecular_hamiltonian, FermionicOperator
from superfermion.chemistry.ansatz import uccsd_ansatz

# Pre-built molecule (cached H2 data — no PySCF needed)
h2 = get_molecular_hamiltonian("H2", basis="sto-3g")
print(f"Terms: {len(h2.terms)}")

# Build UCCSD ansatz (n_qubits, n_electrons)
ansatz = uccsd_ansatz(2, n_electrons=2)
print(f"Ansatz: {ansatz.gate_count} gates, params: {ansatz.parameters}")

# Fermionic operators
fop = FermionicOperator.from_coeffs(np.array([[0.0, 1.0], [1.0, 0.0]]))  # example h1
qubit_op = fop.jordan_wigner(n_qubits=2)
print(f"JW terms: {len(qubit_op.terms)}")

Terms: 5
Ansatz: 3 gates, params: ['theta']
JW terms: 2


In [ ]:
# ── reference/api.mdx #20 ──
import superfermion as sf

# Local tracking (no server needed)
with sf.experiment("bell-study") as tracker:
    r1 = sf.run(sf.Circuit(2).h(0).cnot(0, 1), shots=1000)
    r2 = sf.run(sf.Circuit(3).h(0).cnot(0,1).cnot(1,2), shots=1000)

# Runs saved to ~/.superfermion/runs/bell-study/
print(tracker.runs)  # list of run metadata dicts

[{'device': 'cpu', 'shots': 1000, 'n_qubits': 2, 'depth': 2, 'gate_count': 2, 'started_at': 1786738763.9531868, 'completed_at': 1786738763.953332, 'n_outcomes': 2}, {'device': 'cpu', 'shots': 1000, 'n_qubits': 3, 'depth': 3, 'gate_count': 3, 'started_at': 1786738763.953992, 'completed_at': 1786738763.9541528, 'n_outcomes': 2}]


In [ ]:
# ── reference/api.mdx #21 ──
from superfermion.bridge import (
    from_qiskit, to_qiskit,
    from_qasm, to_qasm,
    from_cirq, to_cirq,
    from_pennylane, to_pennylane,
    to_braket, to_ionq,
)
from superfermion.serialization import to_qasm3, from_qasm3

sf_circuit = sf.Circuit(2).h(0).cnot(0, 1)

# Qiskit interop
qiskit_circuit = to_qiskit(sf_circuit)
sf_circuit = from_qiskit(qiskit_circuit)

# Cirq interop
cirq_circuit = to_cirq(sf_circuit)
sf_circuit = from_cirq(cirq_circuit)

# PennyLane interop — to_pennylane() returns a qfunc; build a tape for the reverse
import pennylane as qml
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def pl_qnode(x):
    qml.RX(x, wires=0)
    qml.CNOT(wires=[0, 1])

pl_tape = qml.workflow.construct_tape(pl_qnode)(0.5)
sf_circuit = from_pennylane(pl_tape)
qfunc = to_pennylane(sf_circuit)

# OpenQASM 2.0
qasm_str = to_qasm(sf_circuit)
sf_circuit = from_qasm(qasm_str)      # Rust-accelerated parser

# OpenQASM 3.0
qasm3_str = sf_circuit.to_qasm3()
sf_circuit = from_qasm3(qasm3_str)

# Braket + IonQ export
braket_circuit = to_braket(sf_circuit)
ionq_circuit = to_ionq(sf_circuit)

# JSON round-trip
json_str = sf_circuit.to_json()
sf_circuit = sf.Circuit.from_json(json_str)
print(sf_circuit.gate_count)

1


In [ ]:
# ── reference/api.mdx #22 ──
from superfermion.mitigation import (
    readout_correction,
    zne,
    zne_with_calibration,
    calibration_based_noise_model,
)

qc = sf.Circuit(2).h(0).cnot(0, 1)

# Readout error correction
result = sf.run(qc, device="cpu", shots=8192)
corrected = readout_correction(result.counts)
print(corrected)

# Zero-noise extrapolation
def energy(sv):
    return sf.expval(sv, sf.PauliString("ZZ", coeff=1.0))

mitigated = zne(qc, energy, scale_factors=[1, 2, 3])
print(f"ZNE <ZZ> = {mitigated:.6f}")

# Build noise model from calibration data
noise_model = calibration_based_noise_model("ibm_like")
result = sf.run(qc, method="density_matrix", noise_model=noise_model, shots=1000)
print(result.counts)

{'11': 4087, '00': 4105}
ZNE <ZZ> = 1.000000
{'11': 497, '00': 503}


In [ ]:
# ── reference/api.mdx #23 ──
# NOT RUN — requires cloud credentials (IBM / IonQ / AWS Braket).
# Doc code verbatim (uncomment + fill in real tokens to run):
# # Standalone provider usage (no platform needed)
# from superfermion.devices.ibm import IBMDevice
# ibm = IBMDevice(token="...")
# result = sf.run(qc, device=ibm("ibm_brisbane"), shots=8192)
print("SKIPPED — requires cloud credentials")

SKIPPED — requires cloud credentials


In [ ]:
# ── reference/api.mdx #24 ──
# Flax/JAX
from superfermion.nn.quantum_layer import QuantumLayer

# PyTorch
from superfermion.nn.torch_layer import TorchQuantumLayer

# TensorFlow
from superfermion.nn.tf_layer import TFQuantumLayer

## Page: `reference/architecture.mdx`

2 Python block(s) — run top-to-bottom.

In [ ]:
# ── reference/architecture.mdx #1 ──
qc = sf.Circuit(2).h(0).cx(0, 1)

In [ ]:
# ── reference/architecture.mdx #2 ──
qc = sf.Circuit(2).h(0).cx(0, 1)

result = sf.run(qc, device="cpu", method="statevector", shots=1024)
print(result.counts)  # result.counts, result.state, result.metadata

state = sf.simulate(qc, device="cpu", method="statevector")
print(state.numpy())  # Returns sf.State directly (shots=0 implied)

{'00': 508, '11': 516}
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
